In [7]:
import msgpackrpc
import threading
import os
import json
import subprocess

In [7]:
env_exec_path_dict = {
    "Japanese_Street": {
        'bash_name': "Japanese_Street.sh",
        'exec_path': "/home/huangth/traveluav_envs/Japanese_Street"
    },
    "London_Street": {
        'bash_name': "London_Street.sh",
        'exec_path': "/home/huangth/traveluav_envs/London_Street"
    },
}

In [3]:
AIRSIM_SETTINGS_JSON = {
    "SettingsVersion": 1.2,
    "SimMode": "Multirotor",
    "PhysicsEngineName": "FastPhysicsEngine", # AirSim自己设计的UAV动力学模型

    # 录制指定相机画面
    "Recording": {
        "RecordOnMove": False,
        "RecordInterval": 0.05,
        "Folder": "/home/user/airsim_rec",
        "Enabled": False,
        "Cameras": [
            { "CameraName": "front_left",  "ImageType": 0, "PixelsAsFloat": False, "Compress": True },
            { "CameraName": "front_left",  "ImageType": 1, "PixelsAsFloat": True,  "Compress": False },
            { "CameraName": "front_right", "ImageType": 0, "PixelsAsFloat": False, "Compress": True }
        ]
    },

    # 设置PX4 UAV
    "Vehicles": {
        "px4_drone": {
            "VehicleType": "PX4Multirotor",
            "AutoCreate": True,
            # 初始位姿
            "X": 0, "Y": 0, "Z": 0,
            "Yaw": 0, "Pitch": 0, "Roll": 0,

            # 机载电脑相机，继承了通用相机参数
            "Cameras": {
                # 仿真 d435i 相机：1个RGB + 2个红外 + 1个红外投射
                # 相机名称只支持 front_center, front_left, front_right, bottom_center, back_center 
                "front_left": { 
                    "X": 0.15, "Y": -0.03, "Z": -0.5, "Pitch": 0, "Roll": 0, "Yaw": 0 ,
                    "CaptureSettings": [
                        {"ImageType": 0, "Width": 848, "Height": 480, "FOV_Degrees": 87},
                        {"ImageType": 1, "Width": 848, "Height": 480, "FOV_Degrees": 87},
                        ]},     # 模拟左IR + depth
                "front_right": { 
                    "X": 0.15, "Y": 0.03, "Z": -0.5, "Pitch": 0, "Roll": 0, "Yaw": 0,
                    "CaptureSettings": [
                        {"ImageType": 0, "Width": 848, "Height": 480, "FOV_Degrees": 87},
                    ]},     # 模拟右IR 
                "front_center": {                                                                       # 模拟RGB
                    "CaptureSettings": [
                        {"ImageType": 0, "Width": 848, "Height": 480, "FOV_Degrees": 87},
                    ],
                    "X": 0.15, "Y": 0.0575, "Z": -0.5, "Pitch": 0, "Roll": 0, "Yaw": 0 },
            },

            # 考虑模型能接受到IMU数据
            "Sensors": {
                "Imu": {
                        "SensorType": 2,
                        "Enabled" : True,
                        "AngularRandomWalk": 0.3,
                        "GyroBiasStabilityTau": 500,
                        "GyroBiasStability": 4.6,
                        "VelocityRandomWalk": 0.24,
                        "AccelBiasStabilityTau": 800,
                        "AccelBiasStability": 36
                }
            }  
        }
    }

}

In [4]:
def kill_pid(port:int, pid: int):
    import signal
    try:
        os.kill(pid, signal.SIGTERM)
        print(f"已杀死占用端口：{port}的进程，PID: {pid}")
    except Exception as e:
        print(f"无法杀死进程 {pid}: {e}")


In [12]:
def from_port_get_pid(port: int):
    import subprocess
    try:
        r = subprocess.run(
            ['lsof', '-t', '-i', f'TCP:{port}', '-sTCP:LISTEN'],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE,
            universal_newlines=True,
        )
        pids = [int(p) for p in r.stdout.split() if int(p) != os.getpid()]
        print(f"当前占用端口 {port} 的进程的PID为: {pids}")
        return pids[0] if pids else None
    except Exception as e:
        print(f"无法获取端口 {port} 的PID: {e}")
        return None

In [ ]:
def clean_ports(ports: list):
    """ 杀死占用 ports 的当前进程"""
    threads = []    
    def _kill_port(port):
        pid = from_port_get_pid(port)
        if pid:
            kill_pid(port, pid)

    for i, port in enumerate(ports):    # 使用多线程同时清理多个端口
        thread = threading.Thread(target=_kill_port, args=(port,), daemon=True)
        threads.append(thread)
    
    for thread in threads:      # 启动所有线程
        thread.start()
    for thread in threads:      # 等待所有线程完成
        thread.join()

    return

# 测试1
测试 kill_pid 和 from_port_get_pid是否能正确占用port，并杀死占用的pid

In [20]:
test_port = 54155
test_process = subprocess.Popen(["python", "-m", "http.server", str(test_port)],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

curr_pid = from_port_get_pid(test_port)
assert curr_pid is not None # 确保该端口被占用
kill_pid(test_port, curr_pid)  # 清理端口
assert from_port_get_pid(test_port) is None # 确保该端口被清理

当前占用端口 54155 的进程的PID为: [8464]
已杀死占用端口：54155的进程，PID: 8464
当前占用端口 54155 的进程的PID为: []


# 测试 2
测试 clean_ports 是否能正确清理端口

In [21]:
test_port_2 = 30000
test_process = subprocess.Popen(["python", "-m", "http.server", str(test_port_2)],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
assert from_port_get_pid(test_port_2) is not None 
clean_ports([test_port_2])  # 清理端口
assert from_port_get_pid(test_port_2) is None # 确保该端口被清理

当前占用端口 30000 的进程的PID为: [8472]
当前占用端口 30000 的进程的PID为: [8472]
已杀死占用端口：30000的进程，PID: 8472
当前占用端口 30000 的进程的PID为: []


In [3]:
import os, json, tempfile, time, socket, copy
import torch

In [9]:
len(env_exec_path_dict)


2

In [ ]:
class SingleSceneHandler(object):
    def __init__(self, env_exec_path_dict, base_port=41451, gpu_ids=[0]):
        """
        创建的时候需要包含所有可用的环境，第一个环境所用的端口，可使用的GPUs
        """
        # TODO: 查看是否存在可使用的gpu_id
        def _pre_check(gpus:list[int]=None, ports:list[int]=None):
            pass
            

        if _pre_check(gpus=gpu_ids):
            self.gpu_ids = gpu_ids
        else:
            raise ValueError(f"存在不支持的GPU！")

        # TODO: 检查是否存在可使用的port，并提示即将杀死该端口上的所有程序
        if _pre_check(ports=base_port):
            self.port_pool = {s: base_port +i for i, s in enumerate(env_exec_path_dict)}
        else:
            raise ValueError(f"从 {base_port} 开始，需要为{len(env_exec_path_dict)}个场景分配端口，但部分端口被占用，请检查！")    

        self.instances = {}        
        self.all_envs = env_exec_path_dict           
        self.base_port = base_port

    def _wait_ready(self, port, proc, timeout=100):
        """ 检查UE子进程是否仍在进行，以及端口是否正在被监听"""
        t0=time.time()

        while time.time() -t0 < timeout:
            if proc.poll() is not None:         # None表示还在跑，否则是退出码
                return False

        with socket.socket() as s:              # 创建一个TCP套接字，用完自动close
            s.settimeout(1)                     # 最多等待1s
            if s.connect_ex(("127.0.0.1", port)) == 0:      # 尝试连接本机端口，0表示成功；非0表示失败
                return True

        time.sleep(2)
        return False

    def _open_scene(self, scene, port, gpu):
        """ 启动"""
        info = {"scene": scene, "port": port, "gpu": gpu,
                "pid": None, "log": None, "ok": False, "error": ""}
        
        try:
            if scene not in env_exec_path_dict:
                raise KeyError(f"未知场景 {scene}")
            cfg = env_exec_path_dict[scene]

            clean_ports([port])                                 # 杀死当前端口上的进程
            self.instances.pop(port, None)                      # 删除该端口上之前运行的进程的记录

            settings = copy.deepcopy(AIRSIM_SETTINGS_JSON)      # 拷贝一份独立的settings文件，保证修改 settings 时不会修改 AIRSIM_SETTINGS_JSON
            settings["ApiServerPort"] = port                    

            f = tempfile.NamedTemporaryFile("w", suffix=f"_{port}.json", delete=False)  # 生成临时的settings文件
            json.dump(settings, f)                                                      # 将当前配置参数写入文件
            f.close()                                           

            log = os.path.join(cfg["exec_path"], f"ue_{port}.log")                      # 在 /home/huangth/traveluav_env/Japanese_Street 目录下创建日志文件 ue_{port}.log
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))

            proc = subprocess.Popen(
                ["bash", cfg["bash_name"], f"-settings={f.name}",
                 "-RenderOffScreen", f"-graphicsadapter={gpu}"],
                cwd=cfg["exec_path"],
                env=env,
                stdout=open(log, "w"),
                stderr=subprocess.STDOUT,
                start_new_session=True
            )                                                                           # 为UE开一个和python进程并行的子进程

            if not self._wait_ready(port, proc):
                raise RuntimeError(f"UE 未就绪，见 {log}")

            self.instances[port] = {"proc": proc, "scene": scene, "gpu": gpu}           # 为该端口添加新纪录
            info.update(pid=proc.pid, log=log, ok=True)

        except Exception as e:
            info["error"] = str(e)
        
        return info

    def open_scene(self, scene:str="Japanese_Street", gpu_id: int=None) -> list:
        """
        启动一个UE场景
        :param scenes   str 

        :return: 返回启动的端口号列表
        """
        # 如果没有指定GPU，则使用第一块GPU
        if gpu_id is None:
            gpu = self.gpu_ids[1]
        else:
            gpu = gpu_id

        info = {
            "scene": scene,
            "gpu_id": gpu,
            "port": None,
            "pid": None,
            "log": None
        }

        try:
            # 场景得要是支持的
            if scene not in self.all_envs:
                raise ValueError(f"Unknown scene: {scene}\n")

            # 获取场景路径和启动文件
            cfg = self.all_envs[scene]

            # 清理端口以及之前的信息
            clean_ports([self.base_port])
            self.instances.pop([self.base_port],None)

            # 为场景生成临时配置文件
            settings = copy.deepcopy(AIRSIM_SETTINGS_JSON)
            f = tempfile.NamedTemporaryFile("w", suffix=f"_{self.base_port}.json", delete=False)
            json.dump(settings, f)
            f.close()

            # 初始化日志文件
            log = os.path.join(cfg["exec_path"], f"ue_{self.base_port}.log")
            
            # 创建运行UE场景的并行子线程
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu_id))
            proc = subprocess.Popen()

            # 检查子线程是否正常工作


            # 记录本地运行

            
        except Exception as e:
            info["error"] = str(e)

        return info   

    def close_scene():
        pass

    def reopen_scene_from_port(self,port: int=None):
        """ 当UE环境崩溃时，自动重启 """

        if port is None:
            raise ValueError(f"Need to specify the port, where a UE env needs to be reopen!\n")
        
        pass

# 测试3
